# 🎛️ Deep Learning Studio · 2 of 5 — Regression: How Many Croissants Tomorrow? 🥐

**ISOM 839 · Prescriptive Analytics · Suffolk University · Prof. Hasan Arslan**

In Studio 1 the answer was yes/no. Now the answer is **a number**: how many croissants will a bakery sell tomorrow? Then the real question: **how many should it bake?** (Not the same thing!)

**Regression** = the answer is a number (demand, price, delivery time, energy use).

### 🧑‍🍳 The recipe (the same five steps in every Studio notebook)
| Step | Plain English |
|---|---|
| **1 · Numbers** | Turn the data into numbers |
| **2 · Shape** | Pick a network shape that fits the data |
| **3 · Train** | Let it practice: guess, measure the miss, nudge the knobs, repeat |
| **4 · Test** | Grade it on data it has never seen |
| **5 · Decide** | Use the prediction to make a business decision |

**Time:** about 20 minutes. Works with or without a GPU.

| Sign | Means |
|---|---|
| ▶️ **Run me** | press **Shift + Enter**, no typing needed |
| 🤔 **Bet first** | say your guess out loud *before* you run the next cell |
| ✅ **What you should see** | if your output is close to this, move on (small differences are normal) |
| ✏️ **Your turn** | change one thing and re-run |

---
## Setup

In [ ]:
# @title ▶️ Run me: setup (about 20 seconds)
import os, time, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import keras
from keras import layers
from IPython.display import display, HTML
keras.utils.set_random_seed(839)

class Progress(keras.callbacks.Callback):
    # prints one short line every few rounds of practice
    def __init__(self, every=5):
        super().__init__(); self.every = every
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 or (epoch + 1) % self.every == 0:
            print(f'round {epoch + 1:3d}:  ' + '   '.join(f'{k} {v:.3f}' for k, v in logs.items()))

def card(title, rows, color='#0a66c2'):
    body = ''.join(f"<tr><td style='padding:4px 18px 4px 0;color:#555'>{a}</td><td style='padding:4px 0;font-weight:700'>{b}</td></tr>" for a, b in rows)
    display(HTML(f"<div style='font:15px Arial;border-left:6px solid {color};padding:8px 16px;margin:6px 0;max-width:760px'>"
                 f"<div style='font-weight:700;font-size:16px;margin-bottom:4px'>{title}</div><table>{body}</table></div>"))

print('✅ Ready! Keras', keras.__version__)

**✅ What you should see:** `✅ Ready! Keras 3.…`

---
## Step 1 · Numbers — 2,500 days at the bakery

Each row is one day: the weekday, the temperature, rain, a promotion, a holiday, and **how many croissants were sold**. A hidden rule creates the sales (weekends are busy, rain keeps weekday customers home, warm dry weekends bring a patio crowd…). The network only sees the days.

In [ ]:
# @title ▶️ Run me: create the bakery's sales history
def make_bakery(n_days=2500, seed=839):
    rng = np.random.default_rng(seed)
    day = np.arange(n_days); weekday = day % 7                      # 0 = Monday ... 6 = Sunday
    temp = np.round(52 + 22 * np.sin(2 * np.pi * (day % 365) / 365 - 1.9) + rng.normal(0, 7, n_days))
    rain = (rng.random(n_days) < 0.28).astype(int)
    promo = (rng.random(n_days) < 0.12).astype(int)
    holiday = (rng.random(n_days) < 0.03).astype(int)
    weekend = weekday >= 5
    # 🙈 the hidden rule
    typical = ((110 + 55 * weekend + 12 * (weekday == 4)) * np.exp(-((temp - 62) / 55) ** 2)
               * (1 - 0.22 * rain * ~weekend - 0.08 * rain * weekend) * (1 + 0.30 * promo) * (1 + 0.5 * holiday)
               + 30 * weekend * (temp > 68) * (1 - rain))
    wobble = np.where(weekend | (promo == 1), 0.30, 0.07)           # weekends and promos are much less predictable
    sold = np.round(typical * np.exp(rng.normal(0, wobble) - wobble ** 2 / 2)).astype(int)
    names = np.array(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])[weekday]
    return pd.DataFrame(dict(weekday=names, temp_f=temp, rain=rain, promo=promo, holiday=holiday, croissants_sold=sold))

bakery = make_bakery()
display(bakery.head(8))
print(f'{len(bakery):,} days. Average sold per day: {bakery.croissants_sold.mean():.0f}   (from {bakery.croissants_sold.min()} to {bakery.croissants_sold.max()})')
bakery.groupby('weekday', sort=False)['croissants_sold'].mean().loc[['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']].plot.bar(figsize=(7, 2.8), color='#d9962a', title='Average croissants sold by weekday'); plt.show()

**✅ What you should see:** 8 days of history, `2,500 days. Average sold per day: 115`, and a bar chart where **Saturday and Sunday are much taller**.

Same two jobs as Studio 1: the weekday **word** becomes 7 yes/no columns, and every column gets the **same scale**. One new rule for forecasting:

> **Never grade a forecast on the past.** We practice on the first 2,000 days and take the exam on the **last 500 days**.

In [ ]:
# @title ▶️ Run me: words → numbers, and split by time
X = pd.get_dummies(bakery.drop(columns='croissants_sold'), columns=['weekday'], dtype='float32').astype('float32').values
y = bakery['croissants_sold'].values.astype('float32')

X_train, y_train = X[:2000], y[:2000]          # practice: the first 2,000 days
X_exam,  y_exam  = X[2000:], y[2000:]          # exam: the last 500 days

scaler = layers.Normalization(); scaler.adapt(X_train)
print('Each day is now', X.shape[1], 'numbers:', X_train[0])

**✅ What you should see:** `Each day is now 11 numbers` (temperature, rain, promo, holiday, and 7 weekday switches).

---
## Step 2 · Shape — the same layers, a different ending

Almost the same network as Studio 1. **One change:** the last neuron has **no squashing**. In Studio 1 we wanted a chance between 0 and 1. Here we want any number, like 137 croissants.

| | Studio 1: classification | Studio 2: regression |
|---|---|---|
| Last layer | `Dense(1, activation='sigmoid')` | `Dense(1)` |
| Answer | a chance, 0 to 1 | any number |
| Loss (how wrong) | cross-entropy | average squared miss |

In [ ]:
# @title ▶️ Run me: a recipe for building the forecaster
def build_forecaster():
    return keras.Sequential([
        keras.Input(shape=(11,)),
        scaler,
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1),                           # no squashing: the answer can be any number
    ])

forecaster = build_forecaster()
print('Knobs to tune:', sum(int(np.prod(w.shape)) for w in forecaster.trainable_weights))

**✅ What you should see:** `Knobs to tune: 2881`.

---
## Step 3 · Train — make the average miss small

**Loss = average squared miss** (`mse`). Squaring makes big misses hurt a lot more than small ones.

A small trick: we count croissants **in hundreds** (137 → 1.37) while training. Networks learn faster with small numbers. We multiply by 100 again when we read the forecast.

In [ ]:
# @title ▶️ Run me: train the forecaster (about 20 seconds)
def train(model, loss):
    model.compile(optimizer=keras.optimizers.Adam(0.003), loss=loss)
    stop_early = keras.callbacks.EarlyStopping(patience=25, restore_best_weights=True)
    return model.fit(X_train, y_train / 100, validation_split=0.2, epochs=300, batch_size=64,
                     verbose=0, callbacks=[Progress(10), stop_early])

history = train(forecaster, 'mse')
plt.figure(figsize=(8, 3.2))
plt.plot(history.history['loss'], label='practice set'); plt.plot(history.history['val_loss'], label='practice exam')
plt.yscale('log'); plt.xlabel('round of practice (epoch)'); plt.ylabel('loss (how wrong)'); plt.legend(); plt.title('Watching it learn'); plt.show()

**✅ What you should see:** the loss drops steeply in the first rounds and flattens. Early stopping ends training after a few dozen rounds.

---
## Step 4 · Test — the last 500 days

We compare against the rule many shops really use: **"sell what we sold on the same weekday last week."**

**🤔 Bet first:** how many croissants off is the network on a typical day? 5? 20? 50?

In [ ]:
# @title ▶️ Run me: grade the forecast on the exam days
forecast  = forecaster.predict(X_exam, verbose=0)[:, 0] * 100      # back to croissants
last_week = y[2000 - 7: 2500 - 7]                                  # same weekday, one week earlier

def avg_miss(guess): return float(np.mean(np.abs(guess - y_exam)))
card('Average miss per day, in croissants (lower is better)', [
    ('"Same as last week"', f'{avg_miss(last_week):.1f}'),
    ('Neural network', f'{avg_miss(forecast):.1f}')])

days = np.arange(42)
plt.figure(figsize=(10, 3.4))
plt.plot(days, y_exam[:42], 'o-', color='black', label='actually sold'); plt.plot(days, forecast[:42], 's--', color='#2bb673', label='network forecast')
plt.plot(days, last_week[:42], ':', color='#e2725b', label='"same as last week"')
plt.xlabel('exam day'); plt.ylabel('croissants'); plt.legend(); plt.title('Six weeks of the exam'); plt.show()

**✅ What you should see:** "same as last week" misses by about **33** croissants a day; the network by about **18**. In the chart the green line follows the weekly rhythm and the weather. It can't hit every point: weekends are simply noisy.

---
## Step 5 · Decide — a forecast is not an order

The bakery sells a croissant for **\$4**. It costs **\$1** to make. Leftovers are given away at closing.

- Bake one **too few** → you lose a **\$3** profit.
- Bake one **too many** → you lose the **\$1** it cost.

Running out hurts three times as much. So should the bakery bake exactly the forecast?

**🤔 Bet first:** bake the forecast, more than the forecast, or less?

In [ ]:
# @title ▶️ Run me: profit over the 500 exam days
PRICE, COST = 4.0, 1.0                                    # ✏️ change COST later

def profit(bake):
    return float(np.sum(PRICE * np.minimum(bake, y_exam) - COST * bake))

plans = {'Bake "same as last week"': last_week,
         'Bake the network forecast': forecast,
         'Bake the forecast + 10% just in case': forecast * 1.10}
for name, bake in plans.items():
    print(f'{name:38s} avg baked {np.mean(bake):4.0f}   profit ${profit(bake):9,.0f}')

**✅ What you should see:** the network forecast earns about **\$158,000**, roughly **\$14,000 more** than "same as last week." Adding a flat 10% buffer earns a bit more again (about **\$160,000**), because running out is the expensive mistake.

But a flat 10% everywhere is crude. Weekdays are predictable; weekends are wild. Here is the modern move:

### Train the network on dollars, not on croissants

So far the network was trained to make its **miss** small. The bakery doesn't care about the miss. It cares about **money**. So we write our own loss: *the dollars lost on a day.* Same data, same shape, new goal.

In [ ]:
# @title ▶️ Run me: same network, trained on dollars lost (about 20 seconds)
def dollars_lost(actual, bake):
    short = keras.ops.maximum(actual - bake, 0)           # customers we turned away
    over  = keras.ops.maximum(bake - actual, 0)           # croissants left over
    return keras.ops.mean((PRICE - COST) * short + COST * over)

decision_net = build_forecaster()
train(decision_net, dollars_lost)
bake_plan = decision_net.predict(X_exam, verbose=0)[:, 0] * 100

card('Two networks, same shape, different goals', [
    ('Trained for accuracy: average miss', f'{avg_miss(forecast):.1f} croissants'),
    ('Trained for accuracy: profit', f'${profit(forecast):,.0f}'),
    ('Trained on dollars: average miss', f'{avg_miss(bake_plan):.1f} croissants'),
    ('Trained on dollars: profit', f'${profit(bake_plan):,.0f}')], color='#2bb673')

weekend = np.isin(bakery['weekday'].values[2000:], ['Sat', 'Sun'])
print(f'Extra it bakes above the forecast:  weekdays {np.mean(bake_plan[~weekend] - forecast[~weekend]):+.0f}   weekends {np.mean(bake_plan[weekend] - forecast[weekend]):+.0f}')

**✅ What you should see:** the dollars-trained network is **less accurate** (average miss about **22 to 23** instead of 18) and **more profitable** (about **\$160,700**, roughly **\$2,000 to 3,000 more**). And look at the last line: it adds only a few croissants on weekdays and **a lot on weekends** (roughly +5 to 10 vs +25 to 30). Nobody told it weekends are unpredictable. It learned where a safety buffer pays.

> **The big lesson of this course in one cell:** the most accurate forecast is not the best decision. Researchers call this *decision-focused learning*. Operations people know its ancestor as the **newsvendor** problem.

### ✏️ Your turn
Thin margins: in the profit cell set `COST = 3.0`, then re-run that cell **and** the dollars cell. *(✅ Now a leftover costs \$3 and running out loses only \$1. The dollars-trained network bakes **fewer** than the forecast, about 100 a day instead of 116, and earns about \$3,500 more than baking the forecast. The flat "+10% just in case" now **loses** about \$9,400. A rule of thumb can't adapt; a network trained on dollars does.)* Set it back to 1.0 when you're done.

### 🧠 Check yourself
Answer in your head first, then click to open.

<details><summary><b>1. What one change turns a yes/no network into a how-many network?</b></summary>

**The ending.** Remove the squashing from the last neuron (`Dense(1)` with no `sigmoid`), and measure the miss with the average squared miss instead of cross-entropy.

</details>

<details><summary><b>2. Why did we split the days by time instead of at random?</b></summary>

A forecast is used on the **future**. Grading it on days mixed in among the practice days would be like peeking at the answers.

</details>

<details><summary><b>3. How can a less accurate network make more money?</b></summary>

Accuracy treats one croissant too many and one too few as equally bad. The bakery doesn't: running out loses \$3, a leftover loses \$1. The dollars-trained network leans high on purpose, most of all on unpredictable weekends.

</details>

---
## 🎯 What you just did
| Step | Here |
|---|---|
| **Numbers** | weekday words → 7 switches, same scale, split **by time** |
| **Shape** | `Dense` layers: 11 → 64 → 32 → 1, **no squashing** at the end |
| **Train** | loss = average squared miss, then **your own loss in dollars** |
| **Test** | average miss on the last 500 days vs "same as last week" |
| **Decide** | bake what the dollars-trained network says |

**The frontier:** forecasting now has its own pretrained *foundation models* (for example Chronos, TimesFM) that forecast a series they have never seen, the same "veteran" idea as TabICL for tables.

**Next:** Studio 3 — the data becomes **pictures**. 👕